# Feature reduction with Stockfish

Which columns can go? First the false predictors (known only after the move or the game), then the inspection of what is left: missing values, near-constant columns, correlations. Only training games are looked at. Run `inspection.py` first.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "config.yaml").exists():
    root = root.parent
sys.path.insert(0, str(root))

import importlib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.shared.plot_style import INK, MUTED, WIDTH, save

paths = importlib.import_module("src.02_preprocessing.common.paths")
reduce = importlib.import_module("src.02_preprocessing.with_stockfish.reduce")
false_predictors = importlib.import_module("src.02_preprocessing.common.false_predictors")
FOLDER = paths.WITH_DIR
INSPECTION = FOLDER / "inspection"
FIG_DIR = INSPECTION / "figures"

In [ ]:
columns = pd.read_csv(INSPECTION / "columns.csv")
corr = pd.read_csv(INSPECTION / "correlation.csv", index_col=0)
pairs = pd.read_csv(INSPECTION / "pairs.csv")
categories = pd.read_csv(INSPECTION / "categories.csv")
meta = json.loads((INSPECTION / "inspection.json").read_text(encoding="utf-8"))
meta

## 1. False predictors: removed before anything else

In [ ]:
rows = [{"column": c, "known": when, "reason": why} for c, (when, why) in false_predictors.FALSE_PREDICTORS.items()]
pd.DataFrame(rows)

## 2. The columns that are left

In [ ]:
columns.round(3)

## 3. Correlation between the columns

Spearman correlation (rank based, so curved relations count too). Dark red or dark blue off the diagonal means two columns carry almost the same information.

In [ ]:
size = max(6, 0.22 * len(corr))
fig, ax = plt.subplots(figsize=(size, size * 0.85))
im = ax.imshow(corr.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=6)
ax.set_yticks(range(len(corr)), corr.index, fontsize=6)
ax.grid(False)
removed = set(reduce.REMOVE)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    if label.get_text() in removed:
        label.set_color("#D55E00")
cb = fig.colorbar(im, ax=ax, fraction=.035, pad=.02)
cb.set_label("Spearman correlation")
ax.set_title("columns removed by the reduction in orange", fontsize=8, color=MUTED)
save(fig, FIG_DIR, "correlation_heatmap")
plt.show()

In [ ]:
pairs.round(3)

## 4. Relation to blunders

Correlation of every number column with the target, and the blunder rate per value of the category columns.

In [ ]:
d = columns.dropna(subset=["corr_target"]).sort_values("corr_target")
fig, ax = plt.subplots(figsize=(WIDTH, 0.16 * len(d) + 0.6))
colors = ["#D55E00" if c in removed else "#0072B2" for c in d["column"]]
ax.barh(d["column"], d["corr_target"], color=colors)
ax.axvline(0, color=INK, lw=.6)
ax.set_xlabel("Spearman correlation with blunder")
ax.tick_params(axis="y", labelsize=6)
save(fig, FIG_DIR, "target_correlation")
plt.show()

In [ ]:
categories.round(2)

## 5. Decision: what the reduction removes and why

*domain knowledge*: decided from what the column means; *inspection*: decided from the numbers above.

In [ ]:
decisions = pd.DataFrame([{"column": c, "basis": b, "reason": r} for c, (b, r) in reduce.REMOVE.items()])
kept = [c for c in columns["column"] if c not in removed]
print(f"{len(columns)} columns after the false predictors, {len(decisions)} removed, {len(kept)} kept")
decisions